# EP2 - Notebook definitivo (GPU)

Um notebook só para as duas famílias de modelo do EP2: **fine-tuning** completo (re-treina o encoder) e
**embeddings congelados** (só inferência + regressão logística depois, na CPU). A troca entre as duas é
automática, pelo `PRESET` escolhido na célula de configuração.

Prevê, para cada dobra, as probabilidades fora da dobra (OOF, para o stacking) e as probabilidades do
**teste1** (900 textos, só para comparação estatística com o EP1) e do **teste2** (897 textos, a entrega
oficial). Os dois testes saem do mesmo treino, não custa rodar de novo depois.

**Como executar**

1. *Ambiente de execução > Alterar o tipo de ambiente de execução* > **GPU T4**.
2. Na célula de configuração, escolha o `PRESET` (ver a lista em `RUNS`, mistura presets de fine-tuning e de
   embeddings) e, se for fine-tuning, as dobras desta sessão (`FOLDS_TO_RUN`).
3. *Ambiente de execução > Executar tudo* e autorize o Google Drive.
4. Na primeira vez, envie `train.xlsx`, `test1.xlsx`, `test2.xlsx` e `folds.npy` quando for pedido (ficam em
   `MyDrive/ep2_pln`, e não são pedidos de novo nas próximas sessões).

**Independência do notebook**: ele nunca depende de nada estar pronto no Drive. Presets de fine-tuning que
reaproveitam o encoder com pré-treino adaptativo (DAPT) do EP1 procuram
`MyDrive/ep1_pln/bert-base-portuguese-cased_dapt`; se achar, reaproveita (economiza ~1-2h); se não achar,
treina esse encoder do zero automaticamente e já salva ali para os próximos presets reaproveitarem. O mesmo
vale por dobra: cada dobra concluída vira `MyDrive/ep2_pln/<PRESET>_fold<k>.npz` e, se a sessão cair, rodar
tudo de novo pula as dobras já prontas (perde no máximo a dobra que estava em andamento).

**Ordem recomendada** (o ganho vem da diversidade no stacking, não de um único modelo):

1. `govbert_br` - BERT de textos governamentais brasileiros (melhor casamento de domínio).
2. `bertimbau_large` - o salto de capacidade mais seguro em cima do EP1.
3. `legalbertimbau` e `bertikal` - jurídico/administrativo, descorrelacionados do BERTimbau-base.
4. `bertimbau_base_dapt_rdrop` - mesma receita do EP1 com R-Drop e suavização ordinal (semente nova). Primeiro
   preset que usa o encoder DAPT compartilhado.
5. `albertina100m` - DeBERTa PT-BR, arquitetura diferente.
6. `serafim335`, `e5large`, `bgem3` - embeddings congelados.
7. Se ainda houver cota de GPU: `bertimbau_large_dapt`, `legalbertimbau_large`, `xlmr_large`.

Depois de rodar cada preset, baixe o `.npz` (ou `.npy`) gerado e coloque em `ep2/gpu_runs/` no projeto local;
`importar_gpu.py` converte para o formato do stacking.

In [ ]:
PRESET = "govbert_br"           # escolha um nome de RUNS, definido logo abaixo
FOLDS_TO_RUN = [0, 1, 2, 3, 4]   # dobras que esta sessão treina (só usado em presets de fine-tuning)
DRIVE_DIR = "/content/drive/MyDrive/ep2_pln"
EP1_DAPT_DIR = "/content/drive/MyDrive/ep1_pln/bert-base-portuguese-cased_dapt"  # encoder DAPT compartilhado
SMOKE_TEST = False               # True: roda com poucos exemplos, só para testar o código

# ===== Fine-tuning (re-treina os pesos do encoder) =====
# Presets marcados com ENCODER="AUTO_DAPT" reaproveitam o encoder com pre-treino adaptativo (MLM) do EP1 se
# ele existir em EP1_DAPT_DIR; senão treinam esse encoder do zero automaticamente (ver célula do DAPT).
FINETUNE_PRESETS = {
    # ~2h30: BERTimbau-large, 335M de parametros
    "bertimbau_large": dict(MODEL_NAME="neuralmind/bert-large-portuguese-cased", LR=1.5e-5,
                            BATCH_SIZE=8, GRAD_ACCUM=4, EVAL_BATCH_SIZE=32, LLRD=0.9),
    # ~3h: o mesmo, com pre-treino adaptativo (MLM) nos textos de treino e teste antes do fine-tuning
    "bertimbau_large_dapt": dict(MODEL_NAME="neuralmind/bert-large-portuguese-cased", DAPT=True, LR=1.5e-5,
                                 BATCH_SIZE=8, GRAD_ACCUM=4, EVAL_BATCH_SIZE=32),
    # ~3h: XLM-RoBERTa-large (multilingue, 560M); embeddings congelados para caber na memoria
    "xlmr_large": dict(MODEL_NAME="FacebookAI/xlm-roberta-large", LR=1e-5, BATCH_SIZE=8, GRAD_ACCUM=4,
                       EVAL_BATCH_SIZE=32, FREEZE_EMBEDDINGS=True),
    # ~1h: a receita do EP1 (BERTimbau-base com o DAPT compartilhado) com outra semente
    "bertimbau_base_dapt_s7": dict(MODEL_NAME="neuralmind/bert-base-portuguese-cased", ENCODER="AUTO_DAPT",
                                   SEED=7),
    "bertimbau_base_dapt_s13": dict(MODEL_NAME="neuralmind/bert-base-portuguese-cased", ENCODER="AUTO_DAPT",
                                    SEED=13),
    # ~1h30: mmBERT-base (ModernBERT multilingue) lendo ate 1024 tokens
    "mmbert_base_1024": dict(MODEL_NAME="jhu-clsp/mmBERT-base", MAX_LEN=1024, HEAD_TOKENS=256, LR=3e-5,
                             BATCH_SIZE=8, GRAD_ACCUM=4, EVAL_BATCH_SIZE=32),
    # ~3h (experimental): LLM Qwen3-0.6B com LoRA e cabeca de classificacao no ultimo token
    "qwen3_0p6b_lora": dict(MODEL_NAME="Qwen/Qwen3-0.6B", USE_LORA=True, LR=2e-4, EPOCHS=2, BATCH_SIZE=8,
                            GRAD_ACCUM=4, EVAL_BATCH_SIZE=32, GRAD_CHECKPOINTING=True, WEIGHT_DECAY=0.0),
    # ~5h (experimental): o mesmo com Qwen3-1.7B, 1 epoca
    "qwen3_1p7b_lora": dict(MODEL_NAME="Qwen/Qwen3-1.7B", USE_LORA=True, LR=1e-4, EPOCHS=1, BATCH_SIZE=4,
                            GRAD_ACCUM=8, EVAL_BATCH_SIZE=16, GRAD_CHECKPOINTING=True, WEIGHT_DECAY=0.0),
    # ~1h: BERT do dominio governamental brasileiro (textos administrativos/legais)
    "govbert_br": dict(MODEL_NAME="dccmpmgfinalisticas/GovBERT-BR", LR=2e-5, R_DROP=1.0, LLRD=0.9),
    # ~1h: BERTimbau adaptado a peticoes e decisoes (Legal-BERTimbau)
    "legalbertimbau": dict(MODEL_NAME="rufimelo/Legal-BERTimbau-base", LR=2e-5, R_DROP=1.0, LLRD=0.9),
    # ~1h: BERTikal (BERTimbau + textos juridicos brasileiros)
    "bertikal": dict(MODEL_NAME="felipemaiapolo/legalnlp-bert", LR=2e-5, LLRD=0.9),
    # ~1h: Albertina 100M PT-BR (DeBERTa) - arquitetura diferente, bom para o stacking
    # NO_AMP=True: o masked_fill da atencao do DeBERTa estoura o range do fp16 dentro do autocast
    # ("RuntimeError: value cannot be converted to type c10::Half without overflow") - roda em fp32.
    "albertina100m": dict(MODEL_NAME="PORTULAN/albertina-100m-portuguese-ptbr-encoder", LR=2e-5,
                          DAPT=False, LLRD=0.9, NO_AMP=True),
    # ~3h: Albertina 900M PT-BR (DeBERTa grande); embeddings congelados para caber na T4
    "albertina900m": dict(MODEL_NAME="PORTULAN/albertina-900m-portuguese-ptbr-encoder-brwac", LR=1e-5,
                          BATCH_SIZE=4, GRAD_ACCUM=8, EVAL_BATCH_SIZE=16, FREEZE_EMBEDDINGS=True,
                          GRAD_CHECKPOINTING=True, DAPT=False, NO_AMP=True),
    # ~2h30: Legal-BERTimbau-large (mesmo tamanho do BERTimbau-large)
    "legalbertimbau_large": dict(MODEL_NAME="rufimelo/Legal-BERTimbau-large", LR=1.5e-5,
                                 BATCH_SIZE=8, GRAD_ACCUM=4, EVAL_BATCH_SIZE=32),
    # ~1h: receita do EP1 com R-Drop (duas passagens + KL) e suavizacao ordinal
    "bertimbau_base_dapt_rdrop": dict(MODEL_NAME="neuralmind/bert-base-portuguese-cased",
                                      ENCODER="AUTO_DAPT", R_DROP=1.0, ORDINAL_SMOOTH=True,
                                      LLRD=0.9, SEED=21),
    # ~1h: a mesma receita, media dos tokens no lugar do [CLS]
    "bertimbau_base_dapt_meanpool": dict(MODEL_NAME="neuralmind/bert-base-portuguese-cased",
                                         ENCODER="AUTO_DAPT", MEAN_POOL=True, LLRD=0.9, SEED=29),
    # ~1h: ignora rotulos de copias discordantes (usa a maioria do grupo no treino)
    "bertimbau_base_dapt_cleandups": dict(MODEL_NAME="neuralmind/bert-base-portuguese-cased",
                                          ENCODER="AUTO_DAPT", CLEAN_DUPS=True, SEED=33),
}
FT_DEFAULTS = dict(ENCODER=None, DAPT=False, MAX_LEN=512, HEAD_TOKENS=128, EPOCHS=3, LR=2e-5, LLRD=1.0,
                   BATCH_SIZE=16, GRAD_ACCUM=2, EVAL_BATCH_SIZE=64, WARMUP=0.1, WEIGHT_DECAY=0.01,
                   LABEL_SMOOTHING=0.1, MAX_GRAD_NORM=1.0, SEED=42, GRAD_CHECKPOINTING=False,
                   FREEZE_EMBEDDINGS=False, USE_LORA=False, LORA_R=16, LORA_ALPHA=32, LORA_DROPOUT=0.05,
                   DAPT_EPOCHS=2, DAPT_LR=5e-5, DAPT_BATCH_SIZE=8, DAPT_GRAD_ACCUM=4, DAPT_WARMUP=0.06,
                   DAPT_MLM_PROB=0.15, R_DROP=0.0, MEAN_POOL=False, ORDINAL_SMOOTH=False, CLEAN_DUPS=False,
                   NO_AMP=False)

# ===== Embeddings congelados (so inferencia; a classificacao entra depois, na CPU) =====
EMBEDDING_PRESETS = {
    # nome curto: (modelo no Hugging Face, pooling, prefixo de instrucao)
    "qwen3emb06b": dict(MODEL_NAME="Qwen/Qwen3-Embedding-0.6B", POOLING="last",
                        PREFIX="Instruct: Classify the clarity of this response from a public agency to a "
                               "citizen's information request\nQuery:"),
    "e5large": dict(MODEL_NAME="intfloat/multilingual-e5-large-instruct", POOLING="mean",
                    PREFIX="Instruct: Classify the clarity of this response from a public agency to a "
                           "citizen's information request\nQuery:"),
    "bgem3": dict(MODEL_NAME="BAAI/bge-m3", POOLING="cls", PREFIX=""),
    "qwen3emb4b": dict(MODEL_NAME="Qwen/Qwen3-Embedding-4B", POOLING="last",
                       PREFIX="Instruct: Classify the clarity of this response from a public agency to a "
                              "citizen's information request\nQuery:"),
    "serafim335": dict(MODEL_NAME="PORTULAN/serafim-335m-portuguese-pt-sentence-encoder", POOLING="mean",
                       PREFIX=""),
    "serafim100": dict(MODEL_NAME="PORTULAN/serafim-100m-portuguese-pt-sentence-encoder", POOLING="mean",
                       PREFIX=""),
}
EMB_MAX_LEN, EMB_HEAD_TOKENS, EMB_BATCH_SIZE = 512, 128, 16

RUNS = {**{k: dict(kind="finetune", **v) for k, v in FINETUNE_PRESETS.items()},
        **{k: dict(kind="embedding", **v) for k, v in EMBEDDING_PRESETS.items()}}
assert PRESET in RUNS, f"PRESET {PRESET!r} nao existe. Opcoes: {sorted(RUNS)}"
KIND = RUNS[PRESET]["kind"]
print(f"PRESET = {PRESET}  (kind = {KIND})")

## Preparacao comum

Instala dependencias, confere a GPU, monta o Drive e carrega `train.xlsx`, `test1.xlsx`, `test2.xlsx` e
`folds.npy` (pede upload so na primeira vez; depois ficam salvos em `MyDrive/ep2_pln`).

In [ ]:
import sys, os, re, json, math, time, random, gc, shutil, subprocess, importlib.util
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from tqdm.auto import tqdm

needed = {"openpyxl": "openpyxl"}
if RUNS[PRESET].get("USE_LORA"):
    needed["peft"] = "peft"
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

SEED = RUNS[PRESET].get("SEED", 42)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
transformers.logging.set_verbosity_error()

IN_COLAB = "google.colab" in sys.modules
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
if DEVICE.type != "cuda" and not SMOKE_TEST:
    raise RuntimeError("GPU nao encontrada. Ambiente de execucao > Alterar o tipo de ambiente de execucao > GPU T4.")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 2**30:.1f} GB)")
print(f"torch {torch.__version__} | transformers {transformers.__version__}")
print(json.dumps({k: v for k, v in RUNS[PRESET].items()}, indent=1, default=str))

WORK_DIR = "."
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = DRIVE_DIR
os.makedirs(WORK_DIR, exist_ok=True)

DATA_FILES = ["train.xlsx", "test1.xlsx", "test2.xlsx", "folds.npy"]
if IN_COLAB and not all(os.path.exists(os.path.join(WORK_DIR, f)) for f in DATA_FILES):
    from google.colab import files
    print("Envie train.xlsx, test1.xlsx, test2.xlsx e folds.npy")
    for name in files.upload():
        target = re.sub(r" \(\d+\)(?=\.)", "", name)
        shutil.move(name, os.path.join(WORK_DIR, target))

LABELS = ["c1", "c234", "c5"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
train = pd.read_excel(os.path.join(WORK_DIR, "train.xlsx"))
test1 = pd.read_excel(os.path.join(WORK_DIR, "test1.xlsx"))
test2 = pd.read_excel(os.path.join(WORK_DIR, "test2.xlsx"))
X = train["resp_text"].fillna("").astype(str).tolist()
y_num = train["clarity"].map(LABEL2ID).to_numpy()
X_test1 = test1["resp_text"].fillna("").astype(str).tolist()
X_test2 = test2["resp_text"].fillna("").astype(str).tolist()
FOLD_ID = np.load(os.path.join(WORK_DIR, "folds.npy"))
assert len(FOLD_ID) == len(X) == 20092 and len(X_test1) == 900 and len(X_test2) == 897 \
    and set(FOLD_ID) == set(range(5))

if SMOKE_TEST:
    keep = np.concatenate([np.where(FOLD_ID == k)[0][:24] for k in range(5)])
    X, y_num, FOLD_ID = [X[i] for i in keep], y_num[keep], FOLD_ID[keep]
    X_test1, X_test2 = X_test1[:16], X_test2[:16]
FOLDS = [(np.where(FOLD_ID != k)[0], np.where(FOLD_ID == k)[0]) for k in range(5)]
y_train = y_num.copy()
if KIND == "finetune" and RUNS[PRESET].get("CLEAN_DUPS"):
    key = pd.Series(X).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    maj = pd.DataFrame({"k": key, "y": y_num}).groupby("k")["y"].agg(lambda s: s.value_counts().idxmax())
    y_train = key.map(maj).to_numpy()
    n_flip = int((y_train != y_num).sum())
    print(f"CLEAN_DUPS: {n_flip} rotulos de treino trocados pela maioria do grupo")
print("treino:", len(X), "| teste1:", len(X_test1), "| teste2:", len(X_test2),
      "| tamanho das dobras:", [len(va) for _, va in FOLDS])

## Pre-processamento comum

URLs e e-mails viram `[URL]` e `[EMAIL]` e os espacos sao normalizados. Usado tanto no fine-tuning quanto
nos embeddings.

In [ ]:
URL_RE = re.compile(r"(https?://|www\.)\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
SPACE_RE = re.compile(r"\s+")


def clean(text):
    text = URL_RE.sub(" [URL] ", str(text))
    text = EMAIL_RE.sub(" [EMAIL] ", text)
    return SPACE_RE.sub(" ", text).strip()

## Caminho A - Fine-tuning (presets com `kind="finetune"`)

Laco em PyTorch puro: AdamW (sem *weight decay* em vieses e normalizacoes), aquecimento linear seguido de
decaimento linear, *gradient clipping*, fp16 com `GradScaler` e acumulacao de gradiente. Textos maiores que
`MAX_LEN` ficam com os `HEAD_TOKENS` primeiros tokens e o restante do final (*head + tail*). As celulas desta
secao so executam quando `KIND == "finetune"`; em presets de embeddings elas nao fazem nada.

In [ ]:
if KIND == "finetune":
    C = SimpleNamespace(**{**FT_DEFAULTS, **{k: v for k, v in RUNS[PRESET].items() if k != "kind"}})
    RUN_NAME = PRESET
    if C.NO_AMP:
        # alguns modelos (DeBERTa/Albertina) estouram o range do fp16 no masked_fill da atencao dentro
        # do autocast ("value cannot be converted to type c10::Half without overflow"); roda em fp32.
        USE_AMP = False
        print("NO_AMP: treinando em fp32 (evita overflow de fp16 conhecido nesse modelo)")

    def special_wrap(tok):
        """Tokens especiais que o tokenizador poe antes e depois do texto (ex.: [CLS] ... [SEP], <s> ... </s>)."""
        full = tok("texto", add_special_tokens=True)["input_ids"]
        bare = tok("texto", add_special_tokens=False)["input_ids"]
        for s in range(len(full) - len(bare) + 1):
            if full[s:s + len(bare)] == bare:
                return full[:s], full[s + len(bare):]
        raise ValueError("nao foi possivel localizar os tokens especiais")

    tokenizer = transformers.AutoTokenizer.from_pretrained(C.MODEL_NAME)
    IS_DECODER = C.USE_LORA
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    PAD_ID = tokenizer.pad_token_id
    if IS_DECODER:
        WRAP_PRE = tokenizer("Resposta de um orgao publico a um pedido de acesso a informacao:\n",
                             add_special_tokens=False)["input_ids"]
        WRAP_POS = tokenizer("\n\nA clareza desta resposta, na avaliacao do cidadao, foi",
                             add_special_tokens=False)["input_ids"]
    else:
        WRAP_PRE, WRAP_POS = special_wrap(tokenizer)
    N_EXTRA = len(WRAP_PRE) + len(WRAP_POS)
    assert 0 <= C.HEAD_TOKENS <= C.MAX_LEN - N_EXTRA

    def encode(texts):
        raw = tokenizer([clean(t) for t in texts], add_special_tokens=False, verbose=False)["input_ids"]
        budget = C.MAX_LEN - N_EXTRA
        tail = budget - C.HEAD_TOKENS
        out = []
        for ids in raw:
            if len(ids) > budget:
                ids = ids[:C.HEAD_TOKENS] + ids[len(ids) - tail:]
            out.append(WRAP_PRE + ids + WRAP_POS)
        return out, np.array([len(ids) for ids in raw])

    t = time.time()
    ids_all, raw_len = encode(X)
    ids_test1, raw_len_test1 = encode(X_test1)
    ids_test2, raw_len_test2 = encode(X_test2)
    print(f"Tokenizacao: {time.time() - t:.0f}s | exemplo: {tokenizer.decode(ids_all[0][:40])!r}")
    print(f"Tokens por texto: mediana {np.median(raw_len):.0f}, p90 {np.percentile(raw_len, 90):.0f}")
    print(f"Truncados com MAX_LEN={C.MAX_LEN}: treino {(raw_len > C.MAX_LEN - N_EXTRA).mean():.1%} | "
          f"teste1 {(raw_len_test1 > C.MAX_LEN - N_EXTRA).mean():.1%} | "
          f"teste2 {(raw_len_test2 > C.MAX_LEN - N_EXTRA).mean():.1%}")

In [ ]:
if KIND == "finetune":
    def free_gpu():
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    def layer_index(name, n_layers):
        m = re.search(r"\.layers?\.(\d+)\.", name)
        if m:
            return int(m.group(1)) + 1
        return 0 if "embed" in name else n_layers + 1

    def make_optimizer(model, lr, warmup, total_steps, weight_decay, llrd=1.0):
        n_layers = getattr(model.config, "num_hidden_layers", 12)
        groups = {}
        for n, p in model.named_parameters():
            if not p.requires_grad:
                continue
            wd = 0.0 if (n.endswith("bias") or "norm" in n.lower()) else weight_decay
            scale = llrd ** (n_layers + 1 - layer_index(n, n_layers))
            groups.setdefault((wd, scale), []).append(p)
        optimizer = torch.optim.AdamW([{"params": ps, "weight_decay": wd, "lr": lr * s}
                                       for (wd, s), ps in groups.items()], lr=lr)
        scheduler = transformers.get_linear_schedule_with_warmup(optimizer, int(warmup * total_steps), total_steps)
        return optimizer, scheduler

    def optimizer_step(model, optimizer, scheduler, scaler):
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], C.MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    def make_batches(lengths, batch_size, rng=None):
        if rng is None:
            order = np.argsort(lengths, kind="stable")[::-1]
            return [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
        idx = rng.permutation(len(lengths))
        chunk = batch_size * 50
        batches = []
        for s in range(0, len(idx), chunk):
            part = idx[s:s + chunk]
            part = part[np.argsort(lengths[part], kind="stable")[::-1]]
            batches += [part[i:i + batch_size] for i in range(0, len(part), batch_size)]
        return [batches[i] for i in rng.permutation(len(batches))]

    def collate(ids_list, idx):
        seqs = [ids_list[i] for i in idx]
        input_ids = torch.full((len(seqs), max(map(len, seqs))), PAD_ID, dtype=torch.long)
        mask = torch.zeros_like(input_ids)
        for j, s in enumerate(seqs):
            input_ids[j, :len(s)] = torch.tensor(s)
            mask[j, :len(s)] = 1
        return input_ids.to(DEVICE), mask.to(DEVICE)

    def autocast():
        return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

    def dtype_kw(dtype):
        """Tipo dos pesos ao carregar (o transformers 5 usaria o do checkpoint, as vezes bf16, ruim na T4)."""
        new_api = tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (4, 56)
        return {"dtype" if new_api else "torch_dtype": dtype}

    @torch.no_grad()
    def predict_proba(model, ids_list):
        model.eval()
        probs = np.zeros((len(ids_list), len(LABELS)), dtype=np.float32)
        for idx in make_batches(np.array([len(s) for s in ids_list]), C.EVAL_BATCH_SIZE):
            input_ids, mask = collate(ids_list, idx)
            with autocast():
                logits = model_logits(model, input_ids, mask)
            probs[idx] = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        return probs

    def model_logits(model, input_ids, mask):
        """Logits da cabeca de classificacao. Com MEAN_POOL, media dos tokens no lugar do [CLS]."""
        if not C.MEAN_POOL:
            return model(input_ids=input_ids, attention_mask=mask).logits
        hidden = model.base_model(input_ids=input_ids, attention_mask=mask).last_hidden_state
        denom = mask.sum(1, keepdim=True).clamp(min=1).unsqueeze(-1)
        pooled = (hidden * mask.unsqueeze(-1)).sum(1) / denom.squeeze(-1)
        if hasattr(model, "dropout"):
            pooled = model.dropout(pooled)
        return model.classifier(pooled)

    def class_loss(logits, labels):
        """Entropia cruzada; se ORDINAL_SMOOTH, a massa suavizada vai so para a classe vizinha."""
        if not C.ORDINAL_SMOOTH:
            return F.cross_entropy(logits.float(), labels, label_smoothing=C.LABEL_SMOOTHING)
        n = logits.size(0)
        target = torch.full((n, len(LABELS)), C.LABEL_SMOOTHING / (len(LABELS) - 1), device=logits.device)
        target.scatter_(1, labels.unsqueeze(1), 1.0 - C.LABEL_SMOOTHING)
        target[labels == 0, 2] = 0
        target[labels == 2, 0] = 0
        target = target / target.sum(1, keepdim=True)
        logp = F.log_softmax(logits.float(), dim=-1)
        return -(target * logp).sum(1).mean()

    def rdrop_loss(model, input_ids, mask, labels):
        """R-Drop: duas passagens com dropout + media das perdas + KL simetrico."""
        logits1 = model_logits(model, input_ids, mask)
        if C.R_DROP <= 0:
            return class_loss(logits1, labels), logits1
        logits2 = model_logits(model, input_ids, mask)
        ce = 0.5 * (class_loss(logits1, labels) + class_loss(logits2, labels))
        p1, p2 = F.log_softmax(logits1.float(), -1), F.log_softmax(logits2.float(), -1)
        q1, q2 = p1.exp(), p2.exp()
        kl = 0.5 * (F.kl_div(p1, q2, reduction="batchmean") + F.kl_div(p2, q1, reduction="batchmean"))
        return ce + C.R_DROP * kl, logits1

### Pre-treino adaptativo ao dominio (DAPT)

Continua o pre-treino de lingua mascarada (15% dos tokens) nos textos unicos de treino, teste1 e teste2, sem
rotulos, concatenados em blocos de `MAX_LEN` tokens. So para modelos da familia BERT.

Dois usos: `DAPT=True` treina um encoder adaptado **so para este preset** (ex.: `bertimbau_large_dapt`).
`ENCODER="AUTO_DAPT"` busca o encoder **compartilhado** em `EP1_DAPT_DIR` (o mesmo dos presets
`bertimbau_base_dapt_*`); se nao encontrar, treina do zero e ja salva la, para os proximos presets que usam
esse mesmo encoder reaproveitarem sem re-treinar.

In [ ]:
if KIND == "finetune":
    CLS_ID, SEP_ID, MASK_ID = tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.mask_token_id

    def mask_tokens(input_ids, generator):
        input_ids = input_ids.clone()
        labels = input_ids.clone()
        prob = torch.full(input_ids.shape, C.DAPT_MLM_PROB)
        prob[(input_ids == CLS_ID) | (input_ids == SEP_ID)] = 0.0
        masked = torch.bernoulli(prob, generator=generator).bool()
        labels[~masked] = -100
        to_mask = torch.bernoulli(torch.full(input_ids.shape, 0.8), generator=generator).bool() & masked
        input_ids[to_mask] = MASK_ID
        to_random = torch.bernoulli(torch.full(input_ids.shape, 0.5), generator=generator).bool() & masked & ~to_mask
        input_ids[to_random] = torch.randint(len(tokenizer), input_ids.shape, generator=generator)[to_random]
        return input_ids.to(DEVICE), labels.to(DEVICE)

    def mlm_loss(model, input_ids, labels):
        with autocast():
            hidden = model.bert(input_ids=input_ids).last_hidden_state
            sel = labels != -100
            logits = model.cls.predictions(hidden[sel])
        return F.cross_entropy(logits.float(), labels[sel])

    @torch.no_grad()
    def mlm_perplexity(model, blocks):
        model.eval()
        gen = torch.Generator().manual_seed(C.SEED)
        total, n = 0.0, 0
        for s in range(0, len(blocks), C.EVAL_BATCH_SIZE):
            input_ids, labels = mask_tokens(torch.as_tensor(blocks[s:s + C.EVAL_BATCH_SIZE]), gen)
            m = int((labels != -100).sum())
            total += mlm_loss(model, input_ids, labels).item() * m
            n += m
        return math.exp(total / n)

    def mlm_train(model, blocks, epochs, lr, desc, gen, rng):
        n_batches = math.ceil(len(blocks) / C.DAPT_BATCH_SIZE)
        total_steps = math.ceil(n_batches / C.DAPT_GRAD_ACCUM) * epochs
        optimizer, scheduler = make_optimizer(model, lr, C.DAPT_WARMUP, total_steps, C.WEIGHT_DECAY)
        scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
        history = []
        for epoch in range(1, epochs + 1):
            model.train()
            t, loss_sum = time.time(), 0.0
            perm = rng.permutation(len(blocks))
            optimizer.zero_grad(set_to_none=True)
            for step in tqdm(range(1, n_batches + 1), desc=f"{desc} epoca {epoch}", mininterval=10):
                batch = perm[(step - 1) * C.DAPT_BATCH_SIZE: step * C.DAPT_BATCH_SIZE]
                input_ids, labels = mask_tokens(torch.as_tensor(blocks[batch]), gen)
                loss = mlm_loss(model, input_ids, labels)
                scaler.scale(loss / C.DAPT_GRAD_ACCUM).backward()
                if step % C.DAPT_GRAD_ACCUM == 0 or step == n_batches:
                    optimizer_step(model, optimizer, scheduler, scaler)
                loss_sum += loss.item()
            history.append({"epoca": epoch, "loss MLM": round(loss_sum / n_batches, 4),
                            "min": round((time.time() - t) / 60, 1)})
            print(desc, history[-1])
        del optimizer, scheduler, scaler
        return history

    def run_dapt(target_dir):
        """Treina o DAPT (MLM) a partir de C.MODEL_NAME e salva em target_dir."""
        t0 = time.time()
        info_file = os.path.join(target_dir, "dapt_info.json")
        texts = pd.Series(X + X_test1 + X_test2).str.strip().drop_duplicates().sample(frac=1, random_state=C.SEED)
        raw = tokenizer([clean(t) for t in texts], add_special_tokens=False, verbose=False)["input_ids"]
        stream = np.concatenate([np.array(ids + [SEP_ID], dtype=np.int64) for ids in raw])
        size = C.MAX_LEN - 2
        n = len(stream) // size
        blocks = np.hstack([np.full((n, 1), CLS_ID), stream[:n * size].reshape(n, size), np.full((n, 1), SEP_ID)])
        n_hold = max(1, int(len(blocks) * 0.02))
        hold, blocks = blocks[:n_hold], blocks[n_hold:]
        print(f"DAPT ({target_dir}): {len(texts)} textos, {len(stream)} tokens, {len(blocks)} blocos de treino")

        model = transformers.AutoModelForPreTraining.from_pretrained(C.MODEL_NAME, **dtype_kw(torch.float32)).to(DEVICE)
        assert hasattr(model, "cls") and hasattr(model, "bert"), "DAPT implementado so para modelos BERT"
        if C.GRAD_CHECKPOINTING:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        gen, rng = torch.Generator().manual_seed(C.SEED), np.random.default_rng(C.SEED)
        ppl_before = mlm_perplexity(model, hold)
        print(f"Perplexidade antes: {ppl_before:.2f}")
        history = []
        if ppl_before > 100:
            print("Perplexidade inicial alta: ajustando so a cabeca de MLM antes do encoder.")
            for p in model.bert.parameters():
                p.requires_grad_(False)
            history += mlm_train(model, blocks[:max(1, len(blocks) // 5)], 1, 1e-3, "cabeca", gen, rng)
            for p in model.bert.parameters():
                p.requires_grad_(True)
            print(f"Perplexidade apos ajustar a cabeca: {mlm_perplexity(model, hold):.2f}")
        history += mlm_train(model, blocks, C.DAPT_EPOCHS, C.DAPT_LR, "DAPT", gen, rng)
        ppl_after = mlm_perplexity(model, hold)
        print(f"Perplexidade depois: {ppl_after:.2f}")
        os.makedirs(target_dir, exist_ok=True)
        model.save_pretrained(target_dir)
        tokenizer.save_pretrained(target_dir)
        info = dict(model=C.MODEL_NAME, ppl_before=ppl_before, ppl_after=ppl_after, history=history,
                    minutes=(time.time() - t0) / 60)
        with open(info_file, "w", encoding="utf-8") as f:
            json.dump(info, f, ensure_ascii=False, indent=1)
        del model
        free_gpu()
        return info

    DAPT_DIR = os.path.join(WORK_DIR, f"{RUN_NAME}_dapt")
    ENCODER = C.ENCODER or C.MODEL_NAME
    if C.DAPT:
        if os.path.exists(os.path.join(DAPT_DIR, "dapt_info.json")):
            print("Encoder adaptado encontrado em", DAPT_DIR)
        else:
            run_dapt(DAPT_DIR)
        ENCODER = DAPT_DIR
    elif C.ENCODER == "AUTO_DAPT":
        if os.path.exists(os.path.join(EP1_DAPT_DIR, "dapt_info.json")):
            print("Encoder DAPT compartilhado encontrado em", EP1_DAPT_DIR, "- reaproveitando.")
        else:
            print("Encoder DAPT compartilhado NAO encontrado em", EP1_DAPT_DIR, "- treinando do zero (mais ~1-2h).")
            run_dapt(EP1_DAPT_DIR)
        ENCODER = EP1_DAPT_DIR
    print("Encoder do fine-tuning:", ENCODER)

### Fine-tuning por dobra

Para cada dobra `k` de `FOLDS_TO_RUN`: treino nas outras 4 dobras por `EPOCHS` epocas fixas e, ao fim de cada
epoca, previsao da dobra `k` (OOF), dos 900 textos do teste1 e dos 897 do teste2. O resultado da dobra vai
para `<PRESET>_fold<k>.npz`; nenhum checkpoint do modelo e guardado (so as probabilidades).

In [ ]:
if KIND == "finetune":
    FOLD_CONFIG = {k: v for k, v in vars(C).items()}
    FOLD_CONFIG["ENCODER"] = ENCODER

    def new_model(encoder):
        kw = dict(num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=LABEL2ID)
        kw.update(dtype_kw(torch.float16 if C.USE_LORA and USE_AMP else torch.float32))
        model = transformers.AutoModelForSequenceClassification.from_pretrained(encoder, **kw)
        if model.config.pad_token_id is None:
            model.config.pad_token_id = PAD_ID
        if C.FREEZE_EMBEDDINGS:
            model.get_input_embeddings().weight.requires_grad_(False)
        if C.GRAD_CHECKPOINTING:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        if C.USE_LORA:
            from peft import LoraConfig, TaskType, get_peft_model
            lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=C.LORA_R, lora_alpha=C.LORA_ALPHA,
                              lora_dropout=C.LORA_DROPOUT,
                              target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj",
                                             "down_proj"])
            model = get_peft_model(model, lora)
            for p in model.parameters():
                if p.requires_grad:
                    p.data = p.data.float()
            model.print_trainable_parameters()
        return model.to(DEVICE)

    def fine_tune(encoder, k, tr_idx, va_idx):
        transformers.set_seed(C.SEED + k)
        model = new_model(encoder)
        train_ids = [ids_all[i] for i in tr_idx]
        train_y = y_train[tr_idx]
        va_ids = [ids_all[i] for i in va_idx]
        lengths = np.array([len(s) for s in train_ids])
        n_batches = math.ceil(len(train_ids) / C.BATCH_SIZE)
        total_steps = math.ceil(n_batches / C.GRAD_ACCUM) * C.EPOCHS
        optimizer, scheduler = make_optimizer(model, C.LR, C.WARMUP, total_steps, C.WEIGHT_DECAY, C.LLRD)
        scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
        rng = np.random.default_rng(C.SEED + k)
        history, epoch_oof, epoch_test1, epoch_test2 = [], [], [], []
        print(f"Dobra {k + 1}: {len(train_ids)} exemplos de treino | {len(va_ids)} OOF | {total_steps} passos")
        for epoch in range(1, C.EPOCHS + 1):
            model.train()
            t0, loss_sum, n_bad = time.time(), 0.0, 0
            optimizer.zero_grad(set_to_none=True)
            batches = make_batches(lengths, C.BATCH_SIZE, rng)
            for step, idx in enumerate(tqdm(batches, desc=f"dobra {k + 1} epoca {epoch}", mininterval=10), 1):
                input_ids, mask = collate(train_ids, idx)
                labels = torch.as_tensor(train_y[idx], device=DEVICE)
                with autocast():
                    loss, logits = rdrop_loss(model, input_ids, mask, labels)
                if not torch.isfinite(loss):
                    n_bad += 1
                    if n_bad > 20:
                        raise RuntimeError("Perda nao finita em muitos passos; reduza LR ou desligue o fp16.")
                    optimizer.zero_grad(set_to_none=True)
                    continue
                scaler.scale(loss / C.GRAD_ACCUM).backward()
                if step % C.GRAD_ACCUM == 0 or step == len(batches):
                    optimizer_step(model, optimizer, scheduler, scaler)
                loss_sum += loss.item()
            P_va = predict_proba(model, va_ids)
            P_te1 = predict_proba(model, ids_test1)
            P_te2 = predict_proba(model, ids_test2)
            epoch_oof.append(P_va)
            epoch_test1.append(P_te1)
            epoch_test2.append(P_te2)
            history.append({"dobra": k + 1, "epoca": epoch, "loss treino": round(loss_sum / len(batches), 4),
                            "min treino": round((time.time() - t0) / 60, 1), "perdas nao finitas": n_bad,
                            "acuracia OOF": round(float(np.mean(P_va.argmax(1) == y_num[va_idx])), 4)})
            print(history[-1])
        del model, optimizer, scheduler, scaler
        free_gpu()
        return np.stack(epoch_oof), np.stack(epoch_test1), np.stack(epoch_test2), history

    def fold_path(k):
        return os.path.join(WORK_DIR, f"{RUN_NAME}_fold{k}.npz")

    def fold_done(k):
        if not os.path.exists(fold_path(k)):
            return False
        z = np.load(fold_path(k))
        return json.loads(str(z["config"])) == json.loads(json.dumps(FOLD_CONFIG)) \
            and np.array_equal(z["idx"], FOLDS[k][1])

    for k in FOLDS_TO_RUN:
        if fold_done(k):
            print(f"Dobra {k + 1}: ja concluida, pulando.")
            continue
        t = time.time()
        tr_idx, va_idx = FOLDS[k]
        result = None
        while result is None:
            try:
                result = fine_tune(ENCODER, k, tr_idx, va_idx)
            except torch.cuda.OutOfMemoryError:
                if C.BATCH_SIZE == 1:
                    raise
                C.BATCH_SIZE //= 2
                C.GRAD_ACCUM *= 2
                C.EVAL_BATCH_SIZE = max(1, C.EVAL_BATCH_SIZE // 2)
            if result is None:
                free_gpu()
                print(f"Sem memoria na GPU: refazendo a dobra com BATCH_SIZE={C.BATCH_SIZE} e "
                      f"GRAD_ACCUM={C.GRAD_ACCUM}")
        epoch_oof, epoch_test1, epoch_test2, history = result
        tmp = fold_path(k)[:-4] + "_tmp.npz"
        np.savez(tmp, idx=va_idx, epoch_oof=epoch_oof, epoch_test1=epoch_test1, epoch_test2=epoch_test2,
                 history=json.dumps(history), config=json.dumps(FOLD_CONFIG))
        os.replace(tmp, fold_path(k))
        print(f"Dobra {k + 1} salva em {fold_path(k)} [{(time.time() - t) / 60:.1f} min]")

### Juncao das 5 dobras

Quando os 5 arquivos `<PRESET>_fold<k>.npz` estao na pasta (de uma ou de varias contas), esta celula monta
`<PRESET>.npz` com OOF (20.092 textos), teste1 (900) e teste2 (897) e faz o download. Esse arquivo vai para
`ep2/gpu_runs/` no projeto local. Mantem os campos `test`/`test_folds` (teste1) com o mesmo nome que o EP1 ja
usava, e acrescenta `test2`/`test2_folds` para a entrega oficial.

In [ ]:
if KIND == "finetune":
    done = [k for k in range(5) if os.path.exists(fold_path(k))]
    if len(done) < 5:
        print(f"Dobras prontas: {done}. Faltam {[k for k in range(5) if k not in done]}.")
        print("Envie os arquivos _fold<k>.npz prontos para quem vai juntar, ou rode as dobras que faltam.")
    else:
        zs = [np.load(fold_path(k)) for k in range(5)]
        n_ep = min(z["epoch_oof"].shape[0] for z in zs)
        epoch_oof = np.zeros((n_ep, len(X), 3), dtype=np.float32)
        for z in zs:
            epoch_oof[:, z["idx"]] = z["epoch_oof"][:n_ep]
        epoch_test1_folds = np.stack([z["epoch_test1"][:n_ep] for z in zs], axis=1)
        epoch_test2_folds = np.stack([z["epoch_test2"][:n_ep] for z in zs], axis=1)
        for e in range(n_ep):
            print(f"epoca {e + 1}: acuracia OOF {np.mean(epoch_oof[e].argmax(1) == y_num):.4f}")
        out = os.path.join(WORK_DIR, f"{RUN_NAME}.npz")
        np.savez(out, fold_id=FOLD_ID, oof=epoch_oof[-1],
                 test=epoch_test1_folds[-1].mean(0), test_folds=epoch_test1_folds[-1],
                 test2=epoch_test2_folds[-1].mean(0), test2_folds=epoch_test2_folds[-1],
                 epoch_oof=epoch_oof, epoch_test1_folds=epoch_test1_folds, epoch_test2_folds=epoch_test2_folds,
                 config=json.dumps(FOLD_CONFIG), history=json.dumps([json.loads(str(z["history"])) for z in zs]))
        print("Salvo em", out)
        if IN_COLAB:
            from google.colab import files
            files.download(out)

## Caminho B - Embeddings congelados (presets com `kind="embedding"`)

Calcula embeddings de treino, teste1 e teste2 com um modelo de *sentence embedding* pre-treinado, **sem
fine-tuning**. A classificacao (regressao logistica nas mesmas 5 dobras) e feita depois, na CPU, em
`importar_gpu.py`. Gera `MyDrive/ep2_pln/emb_<PRESET>.npy` (21.889 linhas: treino, depois teste1, depois
teste2); se o arquivo ja existe, pula direto para o download.

In [ ]:
if KIND == "embedding":
    model_name, pooling, prefix = RUNS[PRESET]["MODEL_NAME"], RUNS[PRESET]["POOLING"], RUNS[PRESET]["PREFIX"]
    texts_all = [clean(t) for t in (X + X_test1 + X_test2)]

    @torch.no_grad()
    def embed(model_name, pooling, prefix, texts):
        tok = transformers.AutoTokenizer.from_pretrained(model_name)
        dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
        new_api = tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (4, 56)
        model = transformers.AutoModel.from_pretrained(
            model_name, **{"dtype" if new_api else "torch_dtype": dtype}).to(DEVICE).eval()
        pre = tok(prefix, add_special_tokens=False)["input_ids"] if prefix else []
        raw = tok(texts, add_special_tokens=False, verbose=False)["input_ids"]
        full = tok("texto", add_special_tokens=True)["input_ids"]
        bare = tok("texto", add_special_tokens=False)["input_ids"]
        s = next(s for s in range(len(full) - len(bare) + 1) if full[s:s + len(bare)] == bare)
        sp_pre, sp_pos = full[:s], full[s + len(bare):]
        wrap = lambda ids: sp_pre + pre + ids + sp_pos
        budget = EMB_MAX_LEN - len(sp_pre) - len(pre) - len(sp_pos)
        seqs = []
        for ids in raw:
            if len(ids) > budget:
                ids = ids[:EMB_HEAD_TOKENS] + ids[len(ids) - (budget - EMB_HEAD_TOKENS):]
            seqs.append(wrap(ids))
        pad = tok.pad_token_id if tok.pad_token_id is not None else 0
        order = np.argsort([len(s) for s in seqs])[::-1]
        out = None
        for s in tqdm(range(0, len(order), EMB_BATCH_SIZE), desc=model_name, mininterval=10):
            idx = order[s:s + EMB_BATCH_SIZE]
            L = max(len(seqs[i]) for i in idx)
            input_ids = torch.full((len(idx), L), pad, dtype=torch.long)
            mask = torch.zeros_like(input_ids)
            for j, i in enumerate(idx):
                if pooling == "last":
                    input_ids[j, L - len(seqs[i]):] = torch.tensor(seqs[i])
                    mask[j, L - len(seqs[i]):] = 1
                else:
                    input_ids[j, :len(seqs[i])] = torch.tensor(seqs[i])
                    mask[j, :len(seqs[i])] = 1
            input_ids, mask = input_ids.to(DEVICE), mask.to(DEVICE)
            h = model(input_ids=input_ids, attention_mask=mask).last_hidden_state.float()
            if pooling == "last":
                e = h[:, -1]
            elif pooling == "cls":
                e = h[:, 0]
            else:
                e = (h * mask[..., None]).sum(1) / mask.sum(1, keepdim=True)
            e = F.normalize(e, dim=-1).cpu().numpy().astype(np.float16)
            if out is None:
                out = np.zeros((len(seqs), e.shape[1]), dtype=np.float16)
            out[idx] = e
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        return out

    path = os.path.join(WORK_DIR, f"emb_{PRESET}.npy")
    if not os.path.exists(path):
        t = time.time()
        E = embed(model_name, pooling, prefix, texts_all)
        bad = int((~np.isfinite(E.astype(np.float32)).all(1)).sum())
        if bad:
            raise RuntimeError(f"{PRESET}: {bad} textos com NaN/inf em fp16")
        np.save(path, E)
        print(f"{PRESET}: {E.shape} [{(time.time() - t) / 60:.1f} min]")
    else:
        print(f"{PRESET}: ja calculado em {path}")
    if IN_COLAB:
        from google.colab import files
        files.download(path)